In [ ]:
import numpy as np


# 1. Génération des données financières

np.random.seed(42) 
n_samples = 100 # nb d'échantillons
n_features = 3  # 3 paramètres pour tester la sélection de variables

X = np.random.randn(n_samples, n_features)

# On définit les vrais coefficients : le paramètre 3 ne sert à rien (0)

true_intercept = 2.0
true_beta = np.array([3.5, -1.5, 0.0])

# Génération de Y avec un bruit gaussien
Y = true_intercept + np.dot(X, true_beta) + np.random.randn(n_samples) * 0.5


# 2. Fonction de seuillage-doux

def seuillage_doux(rho, lam):
    """ C'est le cœur du LASSO qui force les coefficients à zéro """
    if rho < -lam:
        return rho + lam
    elif rho > lam:
        return rho - lam
    else:
        return 0.0

# cf. cours de Stat App   


# 3. Méthode de descente de coordonnées

def lasso_descente_coordonees(X, y, lam, n=500, epsilon=1e-6):
    n_samples, n_features = X.shape
    
    # 1. On initialise l'intercept et les coefficients à zéro
    intercept = 0.0
    beta = np.zeros(n_features)
    
    for it in range(n):
        beta_old = beta.copy()
        
        # Mettre à jour l'intercept (moyenne des résidus)
        intercept = np.mean(y - np.dot(X, beta))
        
        # Mettre à jour chaque coefficient un par un
        for j in range(n_features):
            # Prédiction sans la feature j (en incluant l'intercept actuel)
            y_pred_without_j = intercept + np.dot(X, beta) - beta[j] * X[:, j]
            
            # Résidu (l'erreur que le modèle doit corriger)
            r = y - y_pred_without_j
            
            # rho correspond à la corrélation entre la paramètre j et le résidu
            rho = np.dot(X[:, j], r)
            
            # Application du Seuillage doux
            denominator = np.sum(X[:, j]**2)
            beta[j] = seuillage_doux(rho, lam * n_samples) / denominator
            
        # Condition d'arrêt si l'algorithme a convergé
        if np.linalg.norm(beta - beta_old) < epsilon:
            break
            
    return intercept, beta


# 4. Résultats

lam = 0.1  # Paramètre de régularisation (lambda)

intercept_calcule, beta_calcule = lasso_descente_coordonees(X, Y, lam)

print("="*50)
print("--- COMPARAISON DES RÉSULTATS ---")
print("Vrai Intercept :", true_intercept, " | Calculé :", round(intercept_calcule, 3))
print("Vrais Betas    :", true_beta)
print("Betas LASSO    :", np.round(beta_calcule, 3))
print("="*50)

--- COMPARAISON DES RÉSULTATS ---
Vrai Intercept : 2.0  | Calculé : 2.084
Vrais Betas    : [ 3.5 -1.5  0. ]
Betas LASSO    : [ 3.328 -1.418  0.   ]
